# Día 6 · Unidad 6 — Código profesional (IV): testing con pytest

**Objetivos de aprendizaje**
- Entender por qué la verificación manual de una función (con `print`) no escala, y qué resuelve un framework de tests.
- Escribir tests con `pytest`: `assert`, `pytest.raises`, ejecución con `pytest -vv`.
- Usar *fixtures* (`@pytest.fixture`) para no repetir datos de prueba entre tests, y `conftest.py` para compartirlas entre archivos.
- Categorizar tests con *marks* (`@pytest.mark.unit`, `-m`) y evitar la repetición con `@pytest.mark.parametrize`.
- Pedir a un asistente de IA que genere un test para una función ya escrita, y comprobar que ese test realmente detecta un fallo (no solo que "pasa").

**Duración estimada:** 100-120 minutos — es el bloque con más contenido nuevo del día.

**Requisitos previos:** Día 4 (funciones), `02_teoria_excepciones.ipynb` de este día. Todos los archivos de apoyo están en `pytest_progresion/` (subcarpeta de `dia_06_codigo_profesional/`, junto a este notebook).

## 1. Motivación

Supón que escribes una función que convierte un texto a [snake_case](https://en.wikipedia.org/wiki/Snake_case), en `pytest_progresion/version_1_manual/utils.py`:

```python
def to_snake_case(x: str) -> str:
    return '_'.join(x.lower().split())
```

Una primera forma de comprobar que funciona es, simplemente, imprimir el resultado y mirarlo:

In [ ]:
import sys

sys.path.insert(0, "pytest_progresion/version_1_manual")
from utils import to_snake_case

print(to_snake_case("hola mundo"))
print(to_snake_case("Curso Python"))
print(to_snake_case("notebook"))

Correcto, pero manual: para que alguien más confíe en la función, tendría que leer el código y volver a comprobar los mismos casos a mano. Un primer intento de automatizarlo sería un script de verificación (`verificar_utils.py`) que compara cada resultado con el esperado:

```python
from utils import to_snake_case

def verificar_utils():
    print(to_snake_case("hola mundo") == "hola_mundo")
    print(to_snake_case("Curso Python") == "curso_python")
    print(to_snake_case("notebook") == "notebook")

if __name__ == '__main__':
    verificar_utils()
```

In [ ]:
!cd pytest_progresion/version_1_manual && python verificar_utils.py

Mejor: ahora basta con ejecutar un comando. Pero supón que alguien modifica el script por accidente y cambia una de las entradas (`verificar_utils_2.py`, con `to_snake_case(2)` en vez de `to_snake_case("hola mundo")`):

In [ ]:
!cd pytest_progresion/version_1_manual && python verificar_utils_2.py

El script falla en la primera comprobación con un `AttributeError` (`int` no tiene `.lower()`) y **se detiene ahí** — el resto de comprobaciones ni siquiera se ejecutan. En un proyecto con cientos de funciones probadas así, esto no es sostenible: no sabes cuántos tests más habrían fallado, ni cuáles seguían pasando.

## 2. Uso básico de `pytest`

`pytest` automatiza justo esto: descubre funciones que empiezan por `test_`, las ejecuta todas (aunque alguna falle), y da un resumen claro de cuáles pasaron y cuáles no. El archivo `test_utils.py` de esta misma carpeta (`version_1_manual`) reformula la verificación como tests — deliberadamente con el mismo error que `verificar_utils_2.py`, para ver qué diferencia hace pytest:

```python
import pytest
from utils import to_snake_case

def test_dos_palabras():
    assert to_snake_case(2) == "hola_mundo"          # bug: se pasó un int por error

def test_dos_palabras_mayusculas():
    assert to_snake_case("Curso Python") == "curso_python"

def test_una_palabra():
    assert to_snake_case("notebook") == "notebook"
```

In [ ]:
!cd pytest_progresion/version_1_manual && pytest test_utils.py -vv

A diferencia del script manual, pytest **sí** ejecuta los tres tests aunque el primero falle, y señala con precisión cuál ha fallado (`test_dos_palabras`) y por qué — incluyendo el valor real y el esperado. Los otros dos siguen pasando de forma independiente.

## 3. Refactor con validación explícita

Podemos aprovechar para hacer la función más robusta, validando el tipo de entrada (`pytest_progresion/version_2/utils.py`):

```python
def to_snake_case(x: str) -> str:
    if not isinstance(x, str):
        raise TypeError('El argumento x debe ser un string')
    return '_'.join(x.lower().split())
```

Y sus tests, ya con el caso correcto y uno nuevo para comprobar que se lanza `TypeError` con una entrada inválida (`pytest_progresion/version_2/test_utils.py`):

```python
import pytest
from utils import to_snake_case

def test_dos_palabras():
    assert to_snake_case("hola_mundo") == "hola_mundo"

def test_dos_palabras_mayusculas():
    assert to_snake_case("Curso Python") == "curso_python"

def test_una_palabra():
    assert to_snake_case("notebook") == "notebook"

def test_raises_exception_no_string_arg():
    with pytest.raises(TypeError):
        to_snake_case(5)
```

`pytest.raises(TypeError)` comprueba que el bloque **sí** lanza esa excepción — si no la lanzara, el test fallaría igual que si el resultado fuera incorrecto.

In [ ]:
!cd pytest_progresion/version_2 && pytest -vv

## 4. Fixtures

Imagina que ahora desarrollas, siguiendo el flujo típico de [*test-driven development*](https://en.wikipedia.org/wiki/Test-driven_development) (TDD, escribir el test antes que la implementación), una función `format_data_for_display()` que procesa una lista de personas (nombre, apellido, cargo) y genera cadenas con su nombre completo.

`pytest_progresion/version_3/utils.py`:

```python
from typing import List, Dict

Personas = List[Dict[str, str]]

def format_data_for_display(personas: Personas):
    ...  # implementada más abajo

def format_data_for_csv(personas: Personas):
    # TODO: todavía no implementada — es la fase "roja" de TDD: el test existe antes que el código
    raise NotImplementedError()
```

`pytest_progresion/version_3/test_format_data.py` declara un *fixture* — una función decorada con `@pytest.fixture` que provee los mismos datos de prueba a varios tests, para no repetirlos:

```python
import pytest
from utils import format_data_for_display, format_data_for_csv, Personas


@pytest.fixture
def personas() -> Personas:
    return [
        {"nombre": "Heber", "apellido": "Trujillo", "cargo": "Machine Learning Engineer"},
        {"nombre": "Montserrat", "apellido": "Navarro", "cargo": "Data Engineer"},
    ]

def test_format_data_for_display(personas: Personas):
    assert format_data_for_display(personas) == [
        "Heber Trujillo: Machine Learning Engineer",
        "Montserrat Navarro: Data Engineer",
    ]

def test_format_data_for_csv(personas: Personas):
    assert format_data_for_csv(personas) == (
        "nombre,apellido,cargo\n"
        "Heber,Trujillo,Machine Learning Engineer\n"
        "Montserrat,Navarro,Data Engineer"
    )
```

Cada test que necesita el *fixture* lo recibe simplemente declarándolo como argumento — pytest se encarga de "inyectarlo".

In [ ]:
!cd pytest_progresion/version_3 && pytest -vv

`test_format_data_for_display` pasa; `test_format_data_for_csv` falla con `NotImplementedError`, tal como se espera — la función todavía no está implementada. Esto **no es un error de la migración**: es la fase roja de TDD, deliberada, que se mantiene así en las siguientes versiones porque lo que están demostrando es la evolución de fixtures/marks/parametrización, no completar esa función en concreto.

> **Cuándo evitar fixtures**: son útiles para datos que se repiten en varios tests. Para tests con pequeñas variaciones de datos, forzar un fixture añade una capa de indirección que puede ensuciar más de lo que ordena — la sección 6 (parametrización) es la herramienta correcta para ese caso.

## 5. `conftest.py`: compartir fixtures entre archivos

Si varios archivos `test_*.py` necesitan el mismo *fixture*, moverlo a un módulo `conftest.py` hace que pytest lo ponga a disposición de toda la carpeta automáticamente, sin tener que importarlo en cada archivo (`pytest_progresion/version_4/`):

`conftest.py`:
```python
import pytest
from utils import Personas

@pytest.fixture
def personas() -> Personas:
    return [
        {"nombre": "Heber", "apellido": "Trujillo", "cargo": "Machine Learning Engineer"},
        {"nombre": "Montserrat", "apellido": "Navarro", "cargo": "Data Engineer"},
    ]
```

`test_format_data.py` ya no necesita importar ni definir `personas` — solo lo usa:
```python
from utils import format_data_for_display, format_data_for_csv, Personas

def test_format_data_for_display(personas: Personas):
    ...
```

In [ ]:
!cd pytest_progresion/version_4 && pytest -vv

## 6. Marks: categorizar tests

En una suite grande, no siempre quieres ejecutar todos los tests (por ejemplo, mientras desarrollas una funcionalidad pequeña, o si algunos tests son lentos por acceder a una base de datos). Los *marks* etiquetan tests para poder filtrarlos:

`pytest_progresion/version_5/test_format_data.py`:
```python
import pytest
from utils import format_data_for_display, format_data_for_csv, Personas

@pytest.mark.unit
def test_format_data_for_display(personas: Personas):
    ...

@pytest.mark.e2e
def test_format_data_for_csv(personas: Personas):
    ...
```

Con `-m unit` solo se ejecutan los tests marcados como `unit`:

In [ ]:
!cd pytest_progresion/version_5 && pytest -vv -m unit

> **PRO tip**: como los nombres de los *marks* son libres, es fácil escribir uno mal por error. Para evitarlo, se puede añadir `--strict-markers` a la llamada de pytest, o declarar los marks válidos en la configuración del proyecto (`pyproject.toml`):
> ```toml
> [tool.pytest.ini_options]
> markers = ["unit: pruebas unitarias", "e2e: pruebas de extremo a extremo"]
> ```

## 7. Parametrización

Los *fixtures* no resuelven bien el caso de "el mismo test, con muchas entradas ligeramente distintas". Compara: `pytest_progresion/version_6/utils.py` define `es_palindrome`, y `test_palindrome.py` repite el mismo patrón siete veces:

```python
def test_es_palindrome_string_vacia():
    assert es_palindrome("")

def test_es_palindrome_caracter():
    assert es_palindrome("a")

def test_es_palindrome_mayusculas_minusculas():
    assert es_palindrome("Bob")
# ... y así varias veces más
```

In [ ]:
!cd pytest_progresion/version_6 && pytest -vv

`@pytest.mark.parametrize` reduce todo ese patrón repetido a una sola definición de test que pytest expande automáticamente para cada valor (`pytest_progresion/version_7/test_palindrome.py`):

```python
from utils import es_palindrome
import pytest

@pytest.mark.parametrize("palindrome", [
    "", "a", "Bob", "anita lava la tina", "anita lava la tina?",
])
def test_es_palindrome(palindrome):
    assert es_palindrome(palindrome)

@pytest.mark.parametrize("no_palindrome", ["abc", "abab"])
def test_no_es_palindrome(no_palindrome):
    assert not es_palindrome(no_palindrome)
```

In [ ]:
!cd pytest_progresion/version_7 && pytest -vv

Fíjate en la salida: pytest muestra cada combinación de parámetros como un test independiente (`test_es_palindrome[anita lava la tina]`, etc.) — mismo código, siete casos verificados por separado. El primer argumento de `parametrize()` es el nombre que usará el parámetro dentro del test; el segundo, la lista de valores.

Se puede ir un paso más allá y parametrizar también el resultado esperado, para cubrir casos positivos y negativos en un único test:

```python
@pytest.mark.parametrize("palindrome, resultado_esperado", [
    ("", True),
    ("a", True),
    ("Bob", True),
    ("Never odd or even", True),
    ("abc", False),
    ("abab", False),
])
def test_es_palindrome(palindrome, resultado_esperado):
    assert es_palindrome(palindrome) == resultado_esperado
```

**Para ti:** en `pytest_progresion/version_7/test_palindrome.py` añade un caso más a cada lista de `parametrize` (uno que sí sea palíndromo y uno que no lo sea) y vuelve a ejecutar la celda anterior para comprobar que pytest recoge los nuevos casos.

## 8. Bloque de IA: generar un test con IA y comprobar que detecta fallos de verdad

Pedirle a un asistente de IA que genere un test para una función que ya tienes escrita es uno de los usos más razonables de la IA en testing: es una tarea bastante mecánica una vez que la función y sus casos límite están claros. **Lo que no es mecánico, y sigue siendo tu responsabilidad, es comprobar que el test generado realmente detecta un fallo** — un test que siempre pasa (aunque la función esté rota) no vale nada, y una IA puede generarlo sin darse cuenta.

La función de partida está en `pytest_progresion/ejercicio_ia/utils.py`:

```python
def calcular_prima_total(prima_neta: float, recargo: float) -> float:
    if prima_neta < 0:
        raise ValueError("La prima neta no puede ser negativa")
    if recargo < 0:
        raise ValueError("El recargo no puede ser negativo")
    return prima_neta * (1 + recargo)
```

In [ ]:
prompt_generar_test = """
Contexto: tengo esta función ya escrita y funcionando en pytest_progresion/ejercicio_ia/utils.py,
para un curso de Python de actuarios:

def calcular_prima_total(prima_neta: float, recargo: float) -> float:
    if prima_neta < 0:
        raise ValueError("La prima neta no puede ser negativa")
    if recargo < 0:
        raise ValueError("El recargo no puede ser negativo")
    return prima_neta * (1 + recargo)

Objetivo: genera un archivo test_utils.py con tests de pytest para esta función. Cubre:
1. Un caso normal (prima_neta y recargo positivos), comprobando el valor exacto devuelto.
2. El caso borde recargo=0 (debe devolver la prima_neta sin cambios).
3. Los dos casos en los que debe lanzar ValueError (prima_neta negativa, recargo negativo),
   usando pytest.raises.

Restricciones: usa solo pytest y la función importada de utils, sin dependencias adicionales.
Formato de salida: un único bloque de código Python, listo para guardar como test_utils.py.
"""
print(prompt_generar_test)

**Qué esperar**: una respuesta con 3-4 funciones `test_...` usando `assert` para los casos normales y `pytest.raises(ValueError)` para los casos de error — muy similar en estructura a los tests que has visto en este notebook.

**Qué verificar** (esto es lo que realmente se evalúa, no el prompt): que el test generado **falla cuando debe fallar**. Que pytest lo ejecute en verde no demuestra nada por sí solo — un test mal escrito (por ejemplo, que compare el resultado consigo mismo, o que no llame realmente a la función) también pasaría siempre. La única forma de confiar en un test es romper la función a propósito y comprobar que el test lo detecta.

A continuación, guardamos una versión del test (equivalente a lo que generaría el prompt anterior) y lo verificamos con el propio código, no solo leyéndolo:

In [ ]:
from pathlib import Path

carpeta_ia = Path("pytest_progresion/ejercicio_ia")

test_generado = '''import pytest
from utils import calcular_prima_total


def test_calcular_prima_total_caso_normal():
    assert calcular_prima_total(prima_neta=1000.0, recargo=0.15) == 1150.0


def test_calcular_prima_total_sin_recargo():
    assert calcular_prima_total(prima_neta=500.0, recargo=0.0) == 500.0


def test_calcular_prima_total_prima_negativa():
    with pytest.raises(ValueError):
        calcular_prima_total(prima_neta=-100.0, recargo=0.1)


def test_calcular_prima_total_recargo_negativo():
    with pytest.raises(ValueError):
        calcular_prima_total(prima_neta=100.0, recargo=-0.1)
'''

(carpeta_ia / "test_utils.py").write_text(test_generado, encoding="utf-8")
print("test_utils.py guardado en", carpeta_ia)

In [ ]:
!cd pytest_progresion/ejercicio_ia && pytest -vv

Los cuatro tests pasan con la función correcta. Ahora, la comprobación que de verdad importa: rompemos la función a propósito (cambiamos la multiplicación por una suma, un error de lógica plausible y nada obvio a simple vista) y confirmamos que el test lo detecta:

In [ ]:
utils_path = carpeta_ia / "utils.py"
codigo_original = utils_path.read_text(encoding="utf-8")

codigo_roto = codigo_original.replace(
    "return prima_neta * (1 + recargo)",
    "return prima_neta + recargo",  # bug deliberado: suma en vez de multiplicar
)
assert codigo_roto != codigo_original, "el reemplazo no ha encontrado la línea esperada"

utils_path.write_text(codigo_roto, encoding="utf-8")
print("Función rota a propósito para comprobar el test.")

In [ ]:
!cd pytest_progresion/ejercicio_ia && pytest -vv

Con la función rota, `test_calcular_prima_total_caso_normal` (y `test_calcular_prima_total_sin_recargo`) fallan — el test **sí** detecta el fallo, que es justo lo que había que comprobar. Si en cambio todos los tests hubieran seguido en verde con la función rota, el test generado no serviría, por muy bien que "sonara" al leerlo.

Restauramos la función correcta, para dejar la carpeta como estaba:

In [ ]:
utils_path.write_text(codigo_original, encoding="utf-8")
print("Función restaurada.")

In [ ]:
!cd pytest_progresion/ejercicio_ia && pytest -vv

## Ejercicio del bloque (ampliación)

El material original de este bloque era enteramente demostrativo, sin ningún ejercicio propio del alumno — lo corregimos aquí:

**Para ti:** en `pytest_progresion/ejercicio_ia/utils.py` añade una función nueva `calcular_prima_con_franquicia(prima_total: float, franquicia: float) -> float` que reste la franquicia de la prima total, sin permitir que el resultado sea negativo (si `franquicia > prima_total`, debe devolver `0.0`). Después, en `pytest_progresion/ejercicio_ia/test_utils.py`, añade al menos tres tests: un caso normal, el caso borde `franquicia == prima_total`, y el caso `franquicia > prima_total`. Ejecuta `!cd pytest_progresion/ejercicio_ia && pytest -vv` para comprobar que pasan, y luego rompe tu propia función a propósito para confirmar que al menos un test falla — exactamente igual que se ha hecho arriba.

In [ ]:
# Tu código aquí: puedes escribir la función y los tests directamente en los archivos
# pytest_progresion/ejercicio_ia/utils.py y pytest_progresion/ejercicio_ia/test_utils.py,
# y usar esta celda para ejecutar pytest y comprobar el resultado.


## Resumen

- La verificación manual con `print()` no escala: no queda constancia de qué se comprobó, y un script de verificación se detiene en el primer fallo sin ejecutar el resto.
- `pytest` descubre y ejecuta funciones `test_*`, muestra qué pasó y qué falló de forma independiente, y usa `assert` (más `pytest.raises` para excepciones esperadas).
- Los *fixtures* (`@pytest.fixture`) evitan repetir datos de prueba entre tests; `conftest.py` los comparte entre archivos sin necesidad de importarlos explícitamente.
- Los *marks* (`@pytest.mark.algo`, filtrado con `-m`) categorizan tests para ejecutar solo un subconjunto — típico para separar tests rápidos de lentos.
- `@pytest.mark.parametrize` sustituye varios tests casi idénticos por una única definición expandida automáticamente para cada caso — más DRY y más fácil de ampliar.
- Un test generado por IA (o por ti) solo demuestra algo si **falla cuando la función está rota** — comprobarlo rompiendo la función a propósito no es opcional, es la parte que se evalúa.

**Siguiente:** `05_teoria_git_github.ipynb`, el último bloque de teoría del día: control de versiones con Git y GitHub.